# 01 · Data exploration

Cohort, class balance, the clinical variables that differ between IDH classes, the raw MRI
geometry, and what the encoder actually sees after preprocessing.

**Prerequisite:** run `python prepare_data.py` from the repo root (downloads ~4.6 GB of MRI and
writes `data/cohort.csv`). Nothing here is used as a model input.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # make the `idh` package importable

from collections import Counter

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

from idh.config import ExtractionConfig
from idh.plots import MUTANT_COLOR, WILDTYPE_COLOR, plot_preprocessing
from idh.preprocessing import image_batch, prepare_volume

cfg = ExtractionConfig()
cohort = pd.read_csv(os.environ.get("IDH_COHORT", "../data/cohort.csv"), dtype={"patient_id": str})
train = cohort[cohort["split"] == "train"]
print(len(cohort), "patients")

## Class balance per split

About 1 in 5 patients is IDH-mutant. A model that always answers "wildtype" is therefore
already ~79% accurate, which is why AUROC, sensitivity and balanced accuracy matter more
than plain accuracy here.

In [ ]:
counts = pd.crosstab(cohort["split"], cohort["label"]).reindex(["train", "validation", "test"])
counts.columns = ["wildtype", "mutant"]
counts["mutant_fraction"] = (counts["mutant"] / counts.sum(axis=1)).round(3)
counts

## Clinical variables (training patients)

Age and WHO grade are strongly associated with IDH status: mutant gliomas occur in younger
patients and are mostly grade 2–3. They are **not** model inputs, but they tell us what a
trivial non-imaging predictor can already do. `evaluate.py` therefore reports an age-only
baseline next to the image model.

In [ ]:
display(train.groupby("label")["age"].describe().rename(index={0: "wildtype", 1: "mutant"}))
display(pd.crosstab(train["grade"], train["label"]).rename(columns={0: "wildtype", 1: "mutant"}))
display(pd.crosstab(train["sex"], train["label"]).rename(columns={0: "wildtype", 1: "mutant"}))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
counts[["wildtype", "mutant"]].plot.bar(ax=axes[0], rot=0, color=[WILDTYPE_COLOR, MUTANT_COLOR])
axes[0].set(title="Patients per split", ylabel="Patients", xlabel="")
for label, name, color in [(0, "Wildtype", WILDTYPE_COLOR), (1, "Mutant", MUTANT_COLOR)]:
    axes[1].hist(train.loc[train["label"] == label, "age"], bins=np.arange(0, 101, 10), alpha=0.55,
                 label=name, color=color)
axes[1].set(title="Age at MRI (training patients)", xlabel="Years", ylabel="Patients")
axes[1].legend()
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

### How far does age alone get?

Logistic regression on age, fit on the training patients and scored on validation
(the test split is left for `evaluate.py`).

In [ ]:
from sklearn.metrics import roc_auc_score
from idh.metrics import age_baseline_scores

validation = cohort[cohort["split"] == "validation"]
age_scores, _ = age_baseline_scores(train, validation)
print(f"Age-only AUROC on validation: {roc_auc_score(validation['label'], age_scores):.3f}")

## Raw MRI geometry (training patients)

Reading the NIfTI headers only (no voxel data). UCSF-PDGM volumes are already co-registered,
skull-stripped and resampled to 1 mm isotropic 240 × 240 × 155, so all three contrasts share
one grid.

In [ ]:
rows = []
for row in train.to_dict("records"):
    for modality in cfg.modalities:
        image = nib.load(row[modality])
        rows.append({"patient_id": row["patient_id"], "modality": modality, "shape": tuple(image.shape),
                     "spacing": tuple(round(float(z), 3) for z in image.header.get_zooms()[:3]),
                     "orientation": "".join(nib.aff2axcodes(image.affine))})
headers = pd.DataFrame(rows)
for column in ["shape", "spacing", "orientation"]:
    print(column, Counter(headers[column]).most_common(3))

## Preprocessing: what the encoder sees

Left: the volume after per-contrast percentile normalisation. Right: the cropped, letterboxed
224 × 224 three-channel input (FLAIR, T1c, T2 in place of RGB) for the same slice.

In [ ]:
example = train.iloc[0]
prepared = prepare_volume(example, cfg.modalities, cfg.crop_margin_voxels)
print(f"{example['patient_id']}: {len(prepared['indices'])} nonempty axial slices")
plot_preprocessing(prepared, lambda p, pos: image_batch(p, pos, cfg.image_size), cfg.modalities,
                   title=example["patient_id"]);